In [0]:
'''
Passo: configurar ambiente, imports e parâmetros da live.
O que observar: esta live foca em processamento e visualização; o uso do Volume entra apenas para introduzir o conceito.
Validar: catálogo, schema, volume e path estão apontando para aulas_ao_vivo.live_20260406.deduplicacao_e_chaves.
Sinal de erro: falha ao criar schema/volume ou path fora de /Volumes.
'''

from pyspark.sql import functions as F, Window
from pyspark.sql import types as T
from datetime import datetime, timedelta
import random

CATALOG = 'aulas_ao_vivo'
SCHEMA = 'live_20260406'
VOLUME = 'deduplicacao_e_chaves'

VOLUME_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}'
DATASET_PATH = f'{VOLUME_PATH}/pedidos_mock_parquet'

TOTAL_PEDIDOS_UNICOS = 1_000
TAXA_DUPLICIDADE = 0.30
SEED = 42

'''
ASSUNÇÃO:
- O catálogo `aulas_ao_vivo` já existe no ambiente.
- O schema e o volume podem ser criados neste notebook.
'''

spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`')
spark.sql(f'CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}`') # (Cloud) Object store

print(f'Volume pronto em: {VOLUME_PATH}')
print(f'Dataset será salvo em: {DATASET_PATH}')


In [0]:
'''
Passo: criar a função de geração de dados mockados com taxa de duplicidade parametrizável.
O que observar: a unidade analítica desta live é 1 linha final = 1 pedido único.
Validar: o schema precisa ser aderente ao contrato proposto e a duplicidade deve acontecer sobre `pedido_id`.
Sinal de erro: taxa fora do intervalo de 0 a 1 ou quantidade de pedidos inválida.
'''

def gerar_pedidos_mock(total_pedidos_unicos: int, taxa_duplicidade: float, seed: int = 42):
    if total_pedidos_unicos <= 0:
        raise ValueError('`total_pedidos_unicos` deve ser maior que zero.')

    if not (0 <= taxa_duplicidade <= 1):
        raise ValueError('`taxa_duplicidade` deve estar entre 0 e 1.')

    rng = random.Random(seed)

    produtos = ['notebook', 'monitor', 'teclado', 'mouse', 'headset']
    status_pedido = ['criado', 'pago', 'faturado', 'enviado']
    canais_venda = ['site', 'app', 'marketplace']
    data_base = datetime(2026, 4, 1, 8, 0, 0)

    schema = T.StructType([
        T.StructField('pedido_id', T.StringType(), False),
        T.StructField('cliente_id', T.StringType(), False),
        T.StructField('produto', T.StringType(), False),
        T.StructField('valor_pedido', T.DoubleType(), False),
        T.StructField('status_pedido', T.StringType(), False),
        T.StructField('created_at', T.TimestampType(), False),
        T.StructField('updated_at', T.TimestampType(), False),
        T.StructField('canal_venda', T.StringType(), False)
    ])

    registros_base = []

    for i in range(1, total_pedidos_unicos + 1):
        created_at = data_base + timedelta(minutes=i * 7)
        updated_at = created_at + timedelta(minutes=rng.randint(0, 180))

        registros_base.append((
            f'P{i:06d}',
            f'C{rng.randint(1, max(10, total_pedidos_unicos // 4)):05d}',
            rng.choice(produtos),
            round(rng.uniform(50, 1500), 2),
            rng.choice(status_pedido),
            created_at,
            updated_at,
            rng.choice(canais_venda)
        ))

    total_duplicados = int(total_pedidos_unicos * taxa_duplicidade)
    registros_duplicados = rng.sample(registros_base, total_duplicados) if total_duplicados > 0 else []

    registros_finais = list(registros_base) + list(registros_duplicados)
    rng.shuffle(registros_finais)

    return spark.createDataFrame(registros_finais, schema=schema)


In [0]:
'''
Passo: gerar o dataset sintético, visualizar uma amostra e gravar os dados no Volume em Parquet.
O que observar: a taxa de duplicidade gera linhas extras para alguns `pedido_id`, de forma intencional.
Validar: o total de linhas precisa ser maior que o total distinto de pedidos quando a taxa for maior que zero.
Sinal de erro: gravar fora do path definido ou não perceber a diferença entre linhas totais e pedidos distintos.
'''

pedidos_gerados_df = gerar_pedidos_mock(
    total_pedidos_unicos=TOTAL_PEDIDOS_UNICOS,
    taxa_duplicidade=TAXA_DUPLICIDADE,
    seed=SEED
)

total_linhas_geradas = pedidos_gerados_df.count()
pedidos_distintos_gerados = pedidos_gerados_df.select('pedido_id').distinct().count()
taxa_real_duplicidade = (total_linhas_geradas - pedidos_distintos_gerados) / pedidos_distintos_gerados

print(f'Total de linhas geradas: {total_linhas_geradas}')
print(f'Total de pedidos distintos gerados: {pedidos_distintos_gerados}')
print(f'Taxa real de duplicidade observada: {taxa_real_duplicidade:.2%}')

display(pedidos_gerados_df.orderBy('pedido_id').limit(20))

pedidos_gerados_df.write.mode('overwrite').parquet(DATASET_PATH)
print(f'Dataset gravado com sucesso em: {DATASET_PATH}')


In [0]:
'''
Passo: ler o dataset salvo e executar checagens iniciais.
O que observar: schema, amostra, contagem e nulos ajudam a contextualizar a base antes da investigação.
Validar: os campos principais devem existir e a tabela precisa estar legível a partir do Volume.
Sinal de erro: seguir para a deduplicação sem entender a estrutura básica dos dados.
'''

pedidos = spark.read.parquet(DATASET_PATH)

pedidos.printSchema()
print('Sample do dataframe: ')
display(pedidos.limit(10))

nulos_df = pedidos.select([
    F.sum(F.col(coluna).isNull().cast('int')).alias(coluna)
    for coluna in pedidos.columns
])

display(nulos_df)
print(f'Total de linhas lidas do Volume: {pedidos.count()}')


In [0]:
'''
Passo: medir o tamanho do problema de duplicidade.
O que observar: comparar COUNT com COUNT DISTINCT deixa explícito se a unicidade foi quebrada para a chave escolhida.
Validar: `total_linhas` maior que `pedidos_distintos` indica duplicidade para a unidade analítica desta live.
Sinal de erro: confiar apenas na contagem bruta e seguir a análise como se cada linha fosse um pedido único.
'''

diagnostico_unicidade_df = pedidos.agg(
    F.count('*').alias('total_linhas'),
    F.countDistinct('pedido_id').alias('pedidos_distintos'),
    (F.count('*') - F.countDistinct('pedido_id')).alias('linhas_excedentes')
)

display(diagnostico_unicidade_df)

duplicidades_exatas_df = (
    pedidos
    .groupBy(*pedidos.columns)
    .count()
    .filter(F.col('count') > 1)
    .orderBy(F.col('count').desc(), F.col('pedido_id'))
)

display(duplicidades_exatas_df)


In [0]:
'''
Passo: localizar exatamente onde estão os pedidos duplicados e inspecionar os registros repetidos.
O que observar: a investigação sai do genérico e passa a ser orientada por evidência sobre a chave `pedido_id`.
Validar: os `pedido_id` com `qtde_registros > 1` precisam aparecer claramente para inspeção.
Sinal de erro: remover linhas sem entender quais chaves estão sendo afetadas.
'''

pedidos_duplicados_df = (
    pedidos
    .groupBy('pedido_id')
    .agg(F.count('*').alias('qtde_registros'))
    .filter(F.col('qtde_registros') > 1)
    .orderBy(F.col('qtde_registros').desc(), F.col('pedido_id'))
)

display(pedidos_duplicados_df)

registros_duplicados_detalhe_df = (
    pedidos
    .join(pedidos_duplicados_df.select('pedido_id'), on='pedido_id', how='inner')
    .orderBy('pedido_id', F.col('updated_at').desc())
)

display(registros_duplicados_detalhe_df)


In [0]:
'''
Passo: remover duplicidades com critério explícito usando ROW_NUMBER.
O que observar: a aula pede uma regra determinística; aqui mantemos o registro mais recente por `updated_at`.
Validar: a base final precisa ficar com apenas uma linha por `pedido_id`.
Sinal de erro: usar DISTINCT sem declarar a regra de retenção ou ordenar por uma coluna sem sentido de negócio.
'''

'''
Observação importante:
Como nesta live as duplicidades foram geradas como cópias idênticas, qualquer linha do grupo representa o mesmo dado.
Ainda assim, mantemos a regra por `updated_at` para aderência ao padrão da aula.
'''

janela_deduplicacao = Window.partitionBy('pedido_id').orderBy(F.col('updated_at').desc())

pedidos_deduplicados_df = (
    pedidos
    .withColumn('rn', F.row_number().over(janela_deduplicacao))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

display(pedidos_deduplicados_df.orderBy('pedido_id').limit(20))


In [0]:
'''
Passo: validar se a chave ficou única após a deduplicação.
O que observar: unicidade não deve ser uma suposição; precisa virar evidência com COUNT e COUNT DISTINCT.
Validar: `total_linhas` deve ficar igual a `pedidos_distintos` na base tratada.
Sinal de erro: validar apenas visualmente, sem uma checagem objetiva.
'''

validacao_final_df = pedidos_deduplicados_df.agg(
    F.count('pedido_id').alias('total_linhas'),
    F.countDistinct('pedido_id').alias('pedidos_distintos')
)

display(validacao_final_df)

total_final = pedidos_deduplicados_df.count()
distintos_finais = pedidos_deduplicados_df.select('pedido_id').distinct().count()

assert total_final == distintos_finais, 'A base final ainda possui duplicidade na chave pedido_id.'
print('Validação concluída com sucesso: a chave pedido_id está única na base deduplicada.')


In [0]:
'''
Passo: comparar o impacto direto nas métricas antes e depois da correção.
O que observar: duplicidade infla contagem e também pode inflar receita quando a linha duplicada entra na agregação.
Validar: a diferença entre antes e depois precisa ficar visível para a turma.
Sinal de erro: deduplicar e não mostrar o ganho prático para o negócio.
'''

metricas_antes_df = (
    pedidos
    .agg(
        F.count('*').alias('total_pedidos'),
        F.countDistinct('pedido_id').alias('pedidos_distintos'),
        F.round(F.sum('valor_pedido'), 2).alias('receita_total')
    )
    .withColumn('etapa', F.lit('antes'))
    .select('etapa', 'total_pedidos', 'pedidos_distintos', 'receita_total')
)

metricas_depois_df = (
    pedidos_deduplicados_df
    .agg(
        F.count('*').alias('total_pedidos'),
        F.countDistinct('pedido_id').alias('pedidos_distintos'),
        F.round(F.sum('valor_pedido'), 2).alias('receita_total')
    )
    .withColumn('etapa', F.lit('depois'))
    .select('etapa', 'total_pedidos', 'pedidos_distintos', 'receita_total')
)

comparativo_metricas_df = metricas_antes_df.unionByName(metricas_depois_df)
display(comparativo_metricas_df)

comparativo = {linha['etapa']: linha.asDict() for linha in comparativo_metricas_df.collect()}
inflacao_pedidos = comparativo['antes']['total_pedidos'] - comparativo['depois']['total_pedidos']
inflacao_receita = round(comparativo['antes']['receita_total'] - comparativo['depois']['receita_total'], 2)

print(f'Pedidos inflados pela duplicidade: {inflacao_pedidos}')
print(f'Receita inflada pela duplicidade: R$ {inflacao_receita}')


In [0]:
# spark.sql('DROP SCHEMA IF EXISTS aulas_ao_vivo.live_20260406 CASCADE')